In [1]:
LEAGUE = "brasileirao-serie-a"
SEASON = "2026"
ENVIRONMENT = "dev"
REPROC_MODE = True

In [2]:
import boto3
import json
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("silver") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://cvmc-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

s3 = boto3.client(
    's3',
    endpoint_url='http://cvmc-minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin'
)

print(f"Spark version: {spark.version}")
print("Delta + MinIO configurados!")

Spark version: 3.5.0
Delta + MinIO configurados!


In [3]:
# =====================
# PARÂMETROS DE EXECUÇÃO
# =====================
# LEAGUE = "brasileirao-serie-a"
# SEASON = "2026"
# ENVIRONMENT = "dev"
# REPROC_MODE = False

# =====================
# CONFIGURAÇÕES FIXAS
# =====================
BUCKET = "eng-prd-data-lake"
RAW_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/raw/{LEAGUE}"
SILVER_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/silver/{LEAGUE}"

DEDUP_KEYS = {
    "standing": {
        "pk": ["season", "record.teamId"],
        "order_col": "ingested_at"
    },
    "rounds": {
        "pk": ["record.id"],
        "order_col": "ingested_at"
    },
    "events": {
        "pk": ["record.events.id"],
        "order_col": "ingested_at"
    },
    "players": {
        "pk": ["record.players.id", "record.teams.teamId", "season"],
        "order_col": "ingested_at"
    },
    "team_stats": {
        "pk": ["record.team.teamId", "season"],
        "order_col": "ingested_at"
    },
    "player_stats": {
    "pk": ["record.team.teamId", "season"],
    "order_col": "ingested_at"
    }
}

print(f"League: {LEAGUE} | Season: {SEASON} | Env: {ENVIRONMENT} | Reproc: {REPROC_MODE}")
print(f"Raw:    {RAW_PATH}")
print(f"Silver: {SILVER_PATH}")

League: brasileirao-serie-a | Season: 2026 | Env: dev | Reproc: True
Raw:    s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a
Silver: s3a://eng-prd-data-lake/dev/silver/brasileirao-serie-a


In [3]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

def raw_to_silver(data_type):
    raw = f"{RAW_PATH}/{data_type}"
    silver = f"{SILVER_PATH}/{data_type}"
    config = DEDUP_KEYS[data_type]
    pks = config["pk"]
    order_col = config["order_col"]

    print(f"Lendo {data_type} da Raw...")
    
    df = spark.read.format("delta").load(raw)
    print(raw)
    # Deduplicação — mantém o registro mais recente por chave
    window = Window.partitionBy([col(pk) for pk in pks]) \
                   .orderBy(col(order_col).desc())

    df_dedup = df.withColumn("_rank", row_number().over(window)) \
                 .filter(col("_rank") == 1) \
                 .drop("_rank")

    print(f"Registros após dedup: {df_dedup.count()}")

    df_dedup.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("season") \
        .save(silver)

    print(f"[OK] {data_type} salvo na Silver!\n")

for data_type in ["standing", "rounds", "events", "players", "team_stats", "player_stats"]:
    raw_to_silver(data_type)

print("Camada Silver finalizada!")

Lendo standing da Raw...
s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a/standing
Registros após dedup: 20
[OK] standing salvo na Silver!

Lendo rounds da Raw...
s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a/rounds
Registros após dedup: 8
[OK] rounds salvo na Silver!

Lendo events da Raw...
s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a/events
Registros após dedup: 10
[OK] events salvo na Silver!

Lendo players da Raw...
s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a/players
Registros após dedup: 327
[OK] players salvo na Silver!

Lendo team_stats da Raw...
s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a/team_stats
Registros após dedup: 20
[OK] team_stats salvo na Silver!

Lendo player_stats da Raw...
s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a/player_stats
Registros após dedup: 20
[OK] player_stats salvo na Silver!

Camada Silver finalizada!


In [4]:
df_events = spark.read.format("delta").load(f"{SILVER_PATH}/events")
df_events.printSchema()

root
 |-- record: struct (nullable = true)
 |    |-- awayTeam: struct (nullable = true)
 |    |    |-- categoryId: long (nullable = true)
 |    |    |-- countrySlug: string (nullable = true)
 |    |    |-- foundationdate: string (nullable = true)
 |    |    |-- gender: string (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- internalId: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- national: boolean (nullable = true)
 |    |    |-- shortname: string (nullable = true)
 |    |    |-- slug: string (nullable = true)
 |    |    |-- teamcolorsprimary: string (nullable = true)
 |    |    |-- teamcolorssecondary: string (nullable = true)
 |    |    |-- teamcolorstext: string (nullable = true)
 |    |    |-- tournamentId: long (nullable = true)
 |    |    |-- usercount: long (nullable = true)
 |    |    |-- venueId: long (nullable = true)
 |    |-- events: struct (nullable = true)
 |    |    |-- awayScoreCurrent: long (nullable = tr